# **Optimización TP2**: Optimización con restricciones

In [1]:
using Pkg
Pkg.add(["LinearAlgebra", "Plots"])

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`


In [8]:
include("visualizaciones.jl")
using LinearAlgebra, Plots, IJulia

## Cambio de variables: de $y$ a ángulos $\theta$

### Problema original

$$
\begin{aligned}
\min_{y}\quad & \sum_{i=1}^{n}\left(n-i+\tfrac{1}{2}\right) y_i \\
\text{s.a.}\quad & \sum_{i=1}^{n} y_i = 0 \\
& \sum_{i=1}^{n} \sqrt{1 - y_i^{2}} = 16
\end{aligned}
$$

con la restricción implícita $|y_i| \le 1$ para que la raíz esté definida.

Definimos el ángulo $\theta_i$ de cada eslabón respecto de la horizontal:

$$
y_i = \sin\theta_i,
\qquad
\sqrt{1-y_i^{2}} = \cos\theta_i,
\qquad
\theta_i \in \left[-\tfrac{\pi}{2},\, \tfrac{\pi}{2}\right].
$$

### Problema reformulado

$$
\begin{aligned}
\min_{\theta}\quad & \sum_{i=1}^{n}\left(n-i+\tfrac{1}{2}\right)\sin\theta_i \\
\text{s.a.}\quad & \sum_{i=1}^{n}\sin\theta_i = 0 \\
& \sum_{i=1}^{n}\cos\theta_i = 16
\end{aligned}
$$

Ahora las variables $\theta_i$ son **libres**: la condición $|y_i|\le 1$ se cumple
sola, desaparece la raíz cuadrada y, con ella, sus problemas de dominio. Una vez resuelto, se recupera la
cadena con $y_i = \sin\theta_i$.

## Resolución del problema

In [1]:
# Funcion y gradiente del problema reescrito
f(θ)  = sum((length(θ) - i + 1/2) * sin(θ[i]) for i in eachindex(θ))
∇f(θ) = [(length(θ) - i + 1/2) * cos(θ[i]) for i in eachindex(θ)]

∇f (generic function with 1 method)

In [2]:
# Restricciones y sus gradientes
h1(θ) = sum(sin, θ)
∇h1(θ) =  cos.(θ)

h2(θ) = sum(cos, θ) - 16
∇h2(θ) = -sin.(θ)

∇h2 (generic function with 1 method)

In [3]:
# Función de penalidad y su gradiente
p(θ)  = h1(θ)^2 + h2(θ)^2
∇p(θ) = 2*h1(θ)*∇h1(θ) + 2*h2(θ)*∇h2(θ)

∇p (generic function with 1 method)

In [4]:
# Función a minimizar
q(c, θ)  = f(θ) + c * p(θ)
∇q(c, θ) = ∇f(θ) + c * ∇p(θ)

∇q (generic function with 1 method)

In [5]:
# Descenso por gradiente + Armijo.
function gradiente_armijo(x₀, fun, ∇fun, max_iter=1000)
    # Condiciones iniciales
    xk = x₀
    gk = ∇fun(xk)
    ak = 1/2
    iter = 0
    
    while iter < max_iter
        # Criterio de parada
        if norm(gk) < 1e-6
            break
        end
        
        # Búsqueda del paso (Armijo)
        h(a)    = fun(xk - a .* gk)
        dh₀     = -norm(gk)^2
        armijo  = false
        
        while !armijo
            # Paso muy largo
            if h(ak) > h(0) + dh₀*ak/2
                ak = ak/2
            # Paso muy corto
            elseif h(2*ak) <= h(0) + dh₀*ak
                ak = 1.5*ak
            # Cumple Armijo
            else
                armijo = true
            end
        end
        
        # Actualización
        xk = xk - ak .* gk
        gk = ∇fun(xk)
        iter += 1
    end
    
    return xk
end

gradiente_armijo (generic function with 2 methods)

In [16]:
function min_energia(θ₀, max_iter=50000)
    #=
    Input:
        - θ₀ (Vector{Float64}) := Punto de inicio a optimizar
        - max_iter (Int)       := Opcional, número máximo de iteraciones
    Output:
        - θ_opt (Vector{Float64})             := Punto óptimo del problema
        - iter_vals (Vector{Vector{Float64}}) := Para cada iter_vals[i], se tiene {valor de la función,
                                                valor de la primera restricción, valor de la segunda
                                                restricción y valor de c} en la iteración nro. i
    =#    
    # Condiciones iniciales
    iter = 0
    θk = θ₀
    ck = 1
    iter_vals = Vector{Vector{Float64}}()

    while iter < max_iter
        # Criterio de parada
        if p(θk) < 1e-10
            break
        end

        # Visualización en cada iteración
        IJulia.clear_output(true)
        println("── Iteración $iter | f = $(round(f(θk), digits=3)) | p = $(round(p(θk), digits=3))")
        visualizar_cadena(θk)
        sleep(0.15)

        # Datos de iteración
        push!(iter_vals, [f(θk), h1(θk), h2(θk), ck])

        # Actualización
        ck = 2*ck
        qk(x) = q(ck, x)
        ∇qk(x) = ∇q(ck, x)
        θk = gradiente_armijo(θk, qk, ∇qk)
        iter += 1
    end
    
    return θk, iter_vals
end

min_energia (generic function with 2 methods)

In [17]:
θ_ini = zeros(20)
y_opt, iter_vals = min_energia(y_ini)
println("Finish.")

── Iteración 19 | f = -66.547 | p = 0.0

  Forma de la cadena (vista lateral):

  |O                                          O|
  |                                            |
  | *                                        * |
  |                                            |
  |  *                                      *  |
  |                                            |
  |    *                                  *    |
  |                                            |
  |      *                              *      |
  |                                            |
  |        *                          *        |
  |          *                      *          |
  |             *                *             |
  |                *          *                |
  |                   *  * *                   |
  ----------------------------------------------
  Ganchos: O    Eslabones: *
Finish.
